# Train, Validation, Test Splits, Cross-Validation, and Leakage

> Estimate generalization honestly by respecting the structure and timing of observations.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Separate training, validation, and test roles.
- Use stratification for classification when appropriate.
- Recognize group and time-dependent splits.
- Explain data leakage and why the test set must not guide model selection.

In [1]:
from sklearn.datasets import load_breast_cancer
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GroupKFold, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", y_train.mean().round(3), "Test positive rate:", y_test.mean().round(3))

Train: (455, 30) Test: (114, 30)
Train positive rate: 0.626 Test positive rate: 0.632


## Cross-validation and repeated model selection
Cross-validation estimates variability across several train/validation partitions. It is valuable for small datasets, but it does not automatically fix leakage. Any learned preprocessing must be fit separately within each training fold—normally by using a `Pipeline`.

In [2]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
print("CV ROC-AUC:", scores.round(3))
print("Mean ± std:", round(scores.mean(), 3), "±", round(scores.std(), 3))

CV ROC-AUC: [0.986 0.999 0.997 0.999 0.998]
Mean ± std: 0.996 ± 0.005


## Group and time splits
- **Group split:** keep all records from the same person, hospital, device, or household in one partition when independence requires it.
- **Time split:** train on the past and evaluate on later observations. Randomly shuffling time-series rows can create an unrealistically easy task.
- **Stratification:** approximately preserves class proportions; it does not handle time or group dependence.

In [3]:
# Conceptual examples of valid split strategies
from sklearn.model_selection import GroupShuffleSplit

groups = np.repeat(np.arange(100), 3)
X_dummy = np.arange(len(groups)).reshape(-1, 1)
y_dummy = (X_dummy.ravel() % 7 == 0).astype(int)
gss = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
train_idx, test_idx = next(gss.split(X_dummy, y_dummy, groups=groups))
print("Group overlap:", set(groups[train_idx]) & set(groups[test_idx]))
print("Use TimeSeriesSplit for chronological evaluation; do not shuffle temporal observations by default.")

Group overlap: set()
Use TimeSeriesSplit for chronological evaluation; do not shuffle temporal observations by default.


## A classic leakage mistake
If you impute or scale the full dataset before splitting, information from the test set influences preprocessing. The effect may be subtle, but the test estimate is no longer fully independent. Fit transforms only on the training data, preferably inside the pipeline.

## Practice
1. Pick a split strategy for multiple measurements per customer.
2. Pick a split strategy for next-month sales forecasts.
3. Explain why choosing hyperparameters by repeatedly checking test performance contaminates the test set.